# Dependências

In [1]:
#Prepare credentials to upload table after treatment
#!pip install gcloud
#!gcloud auth application-default login

import pandas as pd
import numpy as np
import time
import os
import pandas_gbq
from google.cloud import bigquery
import glob
import openpyxl
import csv
import re

c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Tratamento xml da FLACSO

In [ ]:
# Read data from an XML file and load it into a pandas DataFrame.
# A DataFrame is a 2-dimensional labeled data structure.
df = pd.read_xml('acoes_afirmativas.xml')

# Rename specific columns of the DataFrame for better readability
# and to follow a consistent naming convention (snake_case).
# The 'inplace=True' argument modifies the DataFrame directly,
# avoiding the need to create a new one.
df.rename(columns={
    "tipoCota": "tipo_cota",
    "pubAlvo": "pub_alvo",
    "comissVer": "comiss_ver"
}, inplace=True)

#Drop specific row
df = df.drop('Unnamed: 0', axis=1)

# Save the modified DataFrame to a CSV (Comma Separated Values) file.
# 'index=False' prevents pandas from writing the DataFrame index as a column in the CSV.
df.to_csv('FLACSO_acoes_afirmativas.csv', index=False)


## Tratamento 2 xml

In [ ]:
# Read data from a CSV file into a pandas DataFrame
# Parameters:
#   encoding="utf8" - ensures proper handling of special characters
#   decimal="," - uses comma as decimal separator for numeric values
df = pd.read_csv('FLACSO_acoes_afirmativas.csv', encoding='utf8', decimal=",")

# Remove unnecessary columns from the DataFrame
# Parameters:
#   ['id', 'marker_id'] - list of columns to drop
#   axis=1 - indicates column-wise operation
#   inplace=True - modifies the DataFrame directly
df.drop(['id', 'marker_id'], axis=1, inplace=True)

# Extract year information from description text and create new 'ano' column
# Method:
#   str.extract() with regex pattern to find dates and capture year component
# Regex explanation:
#   \b(\d{2}\.\d{2}\.(\d{4}))\b matches DD.MM.YYYY format dates
#   [1] selects the second capture group (the year part)
df['ano'] = df['descricao'].str.extract(r'\b(\d{2}\.\d{2}\.(\d{4}))\b')[1]

# Export the processed data to a new CSV file for manual treatment
# Parameters:
#   index=False - prevents writing row numbers to the file
df.to_csv('FLACSO_acoes_afirmativas_v1.csv', index=False)

## Tratamento 2.1 xml

In [ ]:
# Read CSV file with UTF-8 encoding and comma as decimal separator
df = pd.read_csv('FLACSO_acoes_afirmativas_v2.csv', encoding='utf8', decimal=",")

# Rename columns for better clarity and standardization
df.rename(columns={
    'descricao': 'legislacao',
    'regiao': 'nome_regiao',
    'estado': 'sigla_uf',
    'identificacao': 'forma_identificacao',
    'pub_alvo': 'nomenclatura_legislacao',
    'Ano': 'ano',
    'Legislação': 'legislacao',
    'comissionado': 'flag_comissionado',
    'comiss_ver': 'flag_comiss_verificacao',
    'cidade': 'nome_municipio'
}, inplace=True)

# Reorder columns by creating a new column order
temp_cols = df.columns.tolist()
new_cols = temp_cols[4:5] + temp_cols[0:4] + temp_cols[5:]
df = df[new_cols]

# Standardize region names with proper capitalization
df['nome_regiao'] = df['nome_regiao'].replace({
    'sul': 'Sul',
    'norte': 'Norte',
    'nordeste': 'Nordeste',
    'centro-oeste': 'Centro-oeste',
    'sudeste': 'Sudeste'
})

# Standardize 'tipo_cota' values by consolidating similar entries
df['tipo_cota'] = df['tipo_cota'].replace({
    'Concurso público': 'Concurso público',
    'concurso público': 'Concurso público',
    'Concurso Público e estagiário': 'Concurso público e estágio profissional',
    'Concurso público e contratação temporária.': 'Concurso público e contratação temporária',
    'Sistema de pontuação diferenciado em concurso público': 'Concurso público'
})

# Tratamento repositório local

In [2]:
# repositório local
os.chdir('G:\\Drives compartilhados\\República.org\\4. Equipes\\Dados e Comunicação\\DADOS E CONHECIMENTO\\415 - Repositório de Dados\\Repositório Local\\República') 
os.listdir()   # Levantamento próprio de legislações a partir de dados da FLACSO 2024


['Detalhamento_legislacao_assedio.xlsx',
 'rju_legislacao_assedio.csv',
 'assedio_estados.xlsx',
 'Antigos',
 'Publicação no Diário Oficial - Leis assédio',
 'Republica_assedio_leis_estados_atualizado.xlsx',
 'Republica_assedio_estatuto_atualizado.xlsx',
 'Flacso e Republica.org Ações afirmativas.xlsx']

In [3]:
df = pd.read_excel('Flacso e Republica.org Ações afirmativas.xlsx')
df

,ano,cod_uf,sigla_uf,nome_uf,nome_municipio,nome_regiao,possui_cota,esfera,fonte,abrangencia,...,flag_comissionado,percentual_negros,percentual_quilombolas,percentual_indigenas,nomenclatura_legislacao,forma_identificacao,flag_comiss_verificacao,criterios_comissao,vigencia,observacao
0,2003,41,PR,Paraná,Curitiba,Sul,Sim,estadual,https://www.legislacao.pr.gov.br/legislacao/pe...,Geral,...,0.0,10%,Não,Não,Afrodescendentes,Autodeclaração,0.0,Sem informação,vigente,NaN
1,2008,50,MS,Mato Grosso do Sul,Campo Grande,Centro-oeste,Sim,estadual,https://aacpdappls.net.ms.gov.br/appls/legisla...,Geral,...,0.0,20%,Não,3%,Negros e índios,Autodeclaração,0.0,Sem informação,vigente,NaN
2,2011,33,RJ,Rio de Janeiro,Rio de Janeiro,Sudeste,Sim,estadual,http://www3.alerj.rj.gov.br/lotus_notes/defaul...,Geral,...,0.0,20%,Não,englobado nos 20%,Negros e indígenas,Autodeclaração,0.0,Sem informação,vigente,NaN
3,2012,43,RS,Rio Grande do Sul,Porto Alegre,Sul,Sim,estadual,https://ww3.al.rs.gov.br/legis/M010/M0100099.a...,Geral,...,0.0,16%,Não,1%,pessoas negras e integrantes dos povos indígenas,Autodeclaração,1.0,Sim,vigente,A Comissão referida no “caput” deste artigo se...
4,2014,29,BA,Bahia,Salvador,Nordeste,Sim,estadual,https://www.mpba.mp.br/sites/default/files/bib...,Geral,...,0.0,30%,Não,Não,Negros,Autodeclaração,0.0,Sem informação,vigente,NaN
5,2015,16,AP,Amapá,Macapá,Norte,Sim,estadual,https://urano2.mpap.mp.br:8443/download/anexo/...,Geral,...,0.0,20%,Não,Não,Negros,Autodeclaração,0.0,Sem informação,vigente,NaN
6,2015,21,MA,Maranhão,São Luís,Nordeste,Sim,estadual,http://arquivos.al.ma.leg.br:8080/ged/legislac...,Geral,...,0.0,20%,Não,Não,Negros,Autodeclaração,0.0,Sem informação,vigente,NaN
7,2015,35,SP,São Paulo,São Paulo,Sudeste,Sim,estadual,https://www.al.sp.gov.br/repositorio/legislaca...,Geral,...,0.0,Cálculo diferenciado,Não,Cálculo diferenciado,"Pretos, pardos e indígenas",Autodeclaração,1.0,Não,vigente,NaN
8,2017,28,SE,Sergipe,Aracaju,Nordeste,Sim,estadual,https://www.sead.se.gov.br/wp-content/uploads/...,Geral,...,0.0,10%,Não,Não,Afrodescendentes,Autodeclaração,0.0,Sem informação,vigente,NaN
9,2019,53,DF,Distrito Federal,Brasília,Centro-oeste,Sim,estadual,https://www.sinj.df.gov.br/sinj/Norma/cba3dbf7...,Geral,...,0.0,20%,Não,Não,Negros e negras,Autodeclaração,1.0,Sim,vigente,Critérios comissão: A comissão designada para ...


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ano                      27 non-null     int64  
 1   cod_uf                   27 non-null     int64  
 2   sigla_uf                 27 non-null     object 
 3   nome_uf                  27 non-null     object 
 4   nome_municipio           27 non-null     object 
 5   nome_regiao              27 non-null     object 
 6   possui_cota              27 non-null     object 
 7   esfera                   27 non-null     object 
 8   fonte                    19 non-null     object 
 9   abrangencia              19 non-null     object 
 10  legislacao               19 non-null     object 
 11  regulamentacao           19 non-null     object 
 12  tipo_cota                19 non-null     object 
 13  flag_comissionado        19 non-null     float64
 14  percentual_negros        19 

# Upload

In [6]:
schema = [
    bigquery.SchemaField('ano', 'INTEGER', description='Ano de implementação da legislação.'),
    bigquery.SchemaField('cod_uf', 'INTEGER', description='Sigla da Unidade da Federação.'),
    bigquery.SchemaField('sigla_uf', 'STRING', description='Sigla da Unidade da Federação.'),
    bigquery.SchemaField('nome_uf', 'STRING', description='Sigla da Unidade da Federação.'),
    bigquery.SchemaField('nome_municipio', 'STRING', description='Município.'),
    bigquery.SchemaField('nome_regiao', 'STRING', description='Nome da Região.'),
    bigquery.SchemaField('possui_cota', 'STRING', description='Se possui cota.'),
    bigquery.SchemaField('esfera', 'STRING', description='Nível da esfera do governo referente da observação'),
    bigquery.SchemaField('fonte', 'STRING', description='Link da fonte'),
    bigquery.SchemaField('abrangencia', 'STRING', description='Nível de abrangência.'),
    bigquery.SchemaField('legislacao', 'STRING', description='Número da legislação e detalhes.'),   
    bigquery.SchemaField('regulamentacao', 'STRING', description='Detalhes sobre a regulamentação'),
    bigquery.SchemaField('tipo_cota', 'STRING', description='Para quais formas de ingresso a cota vale'),
    bigquery.SchemaField('flag_comissionado', 'INTEGER', description='Se a ação abrange cargos comissionados'),   
    bigquery.SchemaField('percentual_negros', 'STRING', description='Percentual de vagas reservadas para negros'),
    bigquery.SchemaField('percentual_quilombolas', 'STRING', description='Percentual de vagas reservadas para quilombolas'),
    bigquery.SchemaField('percentual_indigenas', 'STRING', description='Percentual de vagas reservadas para indígenas'),
    bigquery.SchemaField('nomenclatura_legislacao', 'STRING', description='Se a abrangência é municipal, estadual ou federal.'),
    bigquery.SchemaField('forma_identificacao', 'STRING', description='Forma de identificação do público alvo na legislação.'),
    bigquery.SchemaField('flag_comiss_verificacao', 'INTEGER', description='Se há comissão de verificação.'),
    bigquery.SchemaField('vigencia', 'STRING', description='Se está vigente ou não.'),
    bigquery.SchemaField('criterios_comissao', 'STRING', description='Critérios de comissão.'),
    bigquery.SchemaField('observacao', 'STRING', description='Observações referente a informação.')
]

client = bigquery.Client(project='repositoriodedadosgpsp')  # Connect to the specified project
dataset_ref = client.dataset('acoes_afirmativas')  # Reference to the target dataset

table_ref = dataset_ref.table('FLACSO_acoes_afirmativas_2025')

job_config = bigquery.LoadJobConfig(schema=schema)

job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result() 

LoadJob<project=repositoriodedadosgpsp, location=US, id=57ad3609-1e54-4348-b713-72dc47059851>